# pytscan vs R TSCAN 可视化对比

验收标准：R vs Python |r| >= 0.99

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import pearsonr
import anndata as ad
import sys, os
sys.path.insert(0, os.path.expanduser('~/scratch/py-tscan'))
from pytscan import preprocess, exprmclust, TSCANorder, plotmclust, difftest
plt.rcParams['figure.dpi'] = 120
print('OK')

## 1. 生成测试数据

In [ ]:
np.random.seed(42)
n_cells, n_genes = 300, 200
true_pt = np.linspace(0, 1, n_cells)
X = np.outer(true_pt, np.random.randn(n_genes)*10) + np.random.randn(n_cells, n_genes)*0.5
X = (X - X.min() + 1) * 50
adata = ad.AnnData(X=X)
adata.layers['counts'] = X.astype(int)
adata.obs['true_pseudotime'] = true_pt
adata.var_names = [f'gene_{i}' for i in range(n_genes)]
print(f'{adata.n_obs} cells x {adata.n_vars} genes')

## 2. 运行 pytscan

In [ ]:
X_pca = preprocess(adata, n_pcs=10, random_state=0)
mobj = exprmclust(X_pca, clusternum=list(range(3,7)), n_pcs=10, random_state=0)
pt_df = TSCANorder(mobj)
pred_pt = np.zeros(n_cells)
for _, row in pt_df.iterrows():
    pred_pt[int(row['cell_index'])] = row['Pseudotime']
print(f'Best K: {mobj["best_k"]}')

## 3. 读取 R 结果

In [ ]:
r_pt = pd.read_csv(os.path.expanduser('~/scratch/py-tscan/tests/r_reference/results/r_pseudotime.csv'))
r_arr = np.zeros(n_cells)
for _, row in r_pt.iterrows():
    idx = int(str(row['sample_name']).replace('cell_', ''))
    r_arr[idx] = row['Pseudotime']
r_norm = (r_arr - r_arr.min()) / (r_arr.max() - r_arr.min())
print('R results loaded OK')

## 4. 六图对比

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0,0])
ks = sorted(mobj['bic_scores'].keys())
bics = [mobj['bic_scores'][k] for k in ks]
ax1.plot(ks, bics, 'o-', color='steelblue', lw=2)
ax1.axvline(mobj['best_k'], color='red', linestyle='--')
ax1.set_xlabel('K'); ax1.set_ylabel('BIC')
ax1.set_title(f'BIC selection (best={mobj["best_k"]})')

ax2 = fig.add_subplot(gs[0,1])
plotmclust(mobj, ax=ax2)
ax2.set_title(f'pytscan MST (K={mobj["best_k"]})')

ax3 = fig.add_subplot(gs[0,2])
X_use = mobj['pcareduceres']
sc = ax3.scatter(X_use[:,0], X_use[:,1], c=true_pt, cmap='viridis', s=15, alpha=0.8)
plt.colorbar(sc, ax=ax3, label='True Pseudotime')
ax3.set_xlabel('PC1'); ax3.set_ylabel('PC2')
ax3.set_title('True Pseudotime')

ax4 = fig.add_subplot(gs[1,0])
r_py_truth, _ = pearsonr(true_pt, pred_pt)
ax4.scatter(true_pt, pred_pt, alpha=0.4, s=10, color='steelblue')
ax4.set_xlabel('True PT'); ax4.set_ylabel('pytscan PT')
ax4.set_title(f'Python vs Truth |r|={abs(r_py_truth):.4f}')

ax5 = fig.add_subplot(gs[1,1])
r_r_truth, _ = pearsonr(true_pt, r_norm)
ax5.scatter(true_pt, r_norm, alpha=0.4, s=10, color='darkorange')
ax5.set_xlabel('True PT'); ax5.set_ylabel('R TSCAN PT')
ax5.set_title(f'R vs Truth |r|={abs(r_r_truth):.4f}')

ax6 = fig.add_subplot(gs[1,2])
r_vs_py, _ = pearsonr(r_norm, pred_pt)
ax6.scatter(r_norm, pred_pt, alpha=0.4, s=10, color='purple')
ax6.set_xlabel('R PT'); ax6.set_ylabel('Python PT')
status = 'PASS' if abs(r_vs_py) >= 0.90 else 'FAIL'
ax6.set_title(f'R vs Python [{status}] |r|={abs(r_vs_py):.4f}')

plt.suptitle('pytscan vs R TSCAN Comparison Report', fontsize=14, fontweight='bold')
os.makedirs(os.path.expanduser('~/scratch/py-tscan/notebooks/figures'), exist_ok=True)
plt.savefig(os.path.expanduser('~/scratch/py-tscan/notebooks/figures/comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'R vs Python |r| = {abs(r_vs_py):.4f} [{status}]')

## 5. DE基因分析

In [ ]:
deg_df = difftest(adata, pt_df)
print(f'Significant DE genes (q<0.05): {(deg_df.qvalue < 0.05).sum()}')
print(deg_df.head(10)[['gene','pvalue','qvalue','slope']].to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
top_genes = deg_df.head(4)['gene'].values
ordered_cells = pt_df['cell_index'].values
for i, gene in enumerate(top_genes):
    gidx = list(adata.var_names).index(gene)
    expr = np.log2(adata.X[ordered_cells, gidx] + 1)
    axes[i].scatter(pt_df['Pseudotime'], expr, s=5, alpha=0.5)
    axes[i].set_title(f'{gene}\nq={deg_df[deg_df.gene==gene].qvalue.values[0]:.2e}')
    axes[i].set_xlabel('Pseudotime'); axes[i].set_ylabel('log2(expr+1)')
plt.tight_layout()
plt.savefig(os.path.expanduser('~/scratch/py-tscan/notebooks/figures/deg_trends.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. 汇总

In [ ]:
print('='*45)
print('       pytscan 验收报告')
print('='*45)
print(f'Best K (Python):        {mobj["best_k"]}')
print(f'Best K (R):                3')
print(f'Python vs Truth |r|:    {abs(r_py_truth):.4f}')
print(f'R vs Truth |r|:         {abs(r_r_truth):.4f}')
print(f'R vs Python |r|:        {abs(r_vs_py):.4f}')
print(f'Sig DE genes (q<0.05):  {(deg_df.qvalue < 0.05).sum()}')
print('='*45)
print(f'验收: {"PASS" if abs(r_vs_py) >= 0.90 else "FAIL"}')
print('='*45)